# Phase C — extend SmolVLA from 20k to 80k

This kernel loads the retained 20k pretrained model from the completed private Phase C SmolVLA kernel and performs 60k additional updates. It starts a new lower-learning-rate optimizer/scheduler instead of restoring the exhausted 20k scheduler.

Required settings: GPU T4 x2, Internet on, HF_TOKEN secret with read access to the private dataset, and the private kernel source zhuokaiyuan/phase-c-smolvla-carton.

Frozen contract: 30 episodes / 6640 frames / 10 Hz; current front and side images, current 6-D state and task text; chunk 50; n_action_steps 1; frozen vision/VLM; action-expert-only training. New checkpoints 10k through 60k correspond to cumulative 30k through 80k.

In [ ]:
import importlib.metadata as metadata
import subprocess
import sys

LEROBOT_COMMIT = "da92db8fc0c935950a56b1ea61fa9b211ef3ac30"
assert sys.version_info >= (3, 12)
package = f"lerobot[training,smolvla] @ git+https://github.com/huggingface/lerobot@{LEROBOT_COMMIT}"
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", package,
        "huggingface_hub==1.19.0", "transformers==5.5.4",
    ],
    check=True,
)
for name in ["lerobot", "torch", "torchvision", "torchcodec", "transformers", "accelerate", "huggingface_hub"]:
    print(f"{name}={metadata.version(name)}")

In [ ]:
# Authenticate, locate the retained 20k model, download the pinned private dataset, and verify both contracts.
import json
import os
from pathlib import Path

SCRATCH_ROOT = Path("/kaggle/temp/phase_c_smolvla_extend")
WORK_ROOT = Path("/kaggle/working/phase_c_smolvla_extend")
DATASET_ROOT = SCRATCH_ROOT / "dataset"
SCRATCH_ROOT.mkdir(parents=True, exist_ok=True)
WORK_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(SCRATCH_ROOT / "hf_home")
os.environ["HF_DATASETS_CACHE"] = str(SCRATCH_ROOT / "hf_home" / "datasets")

from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, login, snapshot_download
from lerobot.datasets.lerobot_dataset import LeRobotDataset
import torch

gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
gpu_memory_gib = [round(torch.cuda.get_device_properties(i).total_memory / 2**30, 2) for i in range(torch.cuda.device_count())]
print("GPUs:", list(zip(gpu_names, gpu_memory_gib, strict=True)))
assert len(gpu_names) == 2 and all("T4" in name for name in gpu_names), "Select GPU T4 x2"

prior_candidates = list(Path("/kaggle/input").glob("**/checkpoints/020000/pretrained_model/model.safetensors"))
assert len(prior_candidates) == 1, f"Expected one retained 20k model, found {prior_candidates}"
MODEL_ROOT = prior_candidates[0].parent
assert prior_candidates[0].stat().st_size > 100_000_000
model_config = json.loads((MODEL_ROOT / "config.json").read_text())
assert model_config["type"] == "smolvla"
assert model_config["chunk_size"] == 50
assert model_config["n_action_steps"] == 1
assert model_config["freeze_vision_encoder"] is True
assert model_config["train_expert_only"] is True
print("20k model:", MODEL_ROOT, f"{prior_candidates[0].stat().st_size / 2**20:.1f} MiB")

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token, add_to_git_credential=False)
api = HfApi(token=hf_token)
DATASET_ID = "stevenzenith/hand_tracking_pv_carton_phase_b"
DATASET_REVISION = "1bb681ab58b5ca2cbdedb52dabf8e1f7a6052a6a"
dataset_info = api.dataset_info(DATASET_ID, revision=DATASET_REVISION)
assert dataset_info.private is True and dataset_info.sha == DATASET_REVISION
snapshot_download(
    repo_id=DATASET_ID, repo_type="dataset", revision=DATASET_REVISION,
    local_dir=DATASET_ROOT, token=hf_token,
)
del hf_token

dataset = LeRobotDataset(
    repo_id=DATASET_ID, root=DATASET_ROOT, revision=DATASET_REVISION, video_backend="torchcodec",
)
assert dataset.num_episodes == 30 and dataset.num_frames == 6640 and dataset.fps == 10
assert set(dataset.meta.camera_keys) == {"observation.images.front", "observation.images.side"}
sample = dataset[0]
assert tuple(sample["observation.images.front"].shape) == (3, 480, 640)
assert tuple(sample["observation.images.side"].shape) == (3, 480, 640)
assert tuple(sample["observation.state"].shape) == (6,) and tuple(sample["action"].shape) == (6,)
print("dataset:", dataset.num_episodes, "episodes /", dataset.num_frames, "frames /", dataset.fps, "Hz")
del dataset, sample

In [ ]:
# Build the two-GPU low-LR continuation command and persistent logger.
import datetime as dt
import shutil
import subprocess

TRAIN_CLI = shutil.which("lerobot-train")
ACCELERATE = shutil.which("accelerate")
assert TRAIN_CLI and ACCELERATE
PER_GPU_BATCH = 2
START_TOTAL_STEPS = 20_000
EXTRA_STEPS = 60_000
RENAME_MAP = json.dumps({
    "observation.images.front": "observation.images.camera1",
    "observation.images.side": "observation.images.camera2",
}, separators=(",", ":"))

def build_command(*, steps: int, output_dir: Path, job_name: str, save_checkpoint: bool, save_freq: int, log_freq: int) -> list[str]:
    warmup_steps = 500 if steps >= EXTRA_STEPS else 20
    decay_steps = EXTRA_STEPS if steps >= EXTRA_STEPS else 20
    train_args = [
        f"--policy.path={MODEL_ROOT}",
        f"--dataset.repo_id={DATASET_ID}", f"--dataset.root={DATASET_ROOT}",
        f"--dataset.revision={DATASET_REVISION}", "--dataset.video_backend=torchcodec",
        f"--rename_map={RENAME_MAP}", f"--batch_size={PER_GPU_BATCH}", "--num_workers=2",
        "--policy.chunk_size=50", "--policy.n_action_steps=1",
        "--policy.freeze_vision_encoder=true", "--policy.train_expert_only=true",
        "--policy.load_vlm_weights=true", "--policy.device=cuda", "--policy.use_amp=false",
        "--policy.push_to_hub=false", "--policy.optimizer_lr=1e-5",
        "--policy.scheduler_decay_lr=2.5e-6",
        f"--policy.scheduler_warmup_steps={warmup_steps}",
        f"--policy.scheduler_decay_steps={decay_steps}", f"--steps={steps}",
        "--eval_freq=0", f"--save_checkpoint={str(save_checkpoint).lower()}",
        f"--save_freq={save_freq}", f"--log_freq={log_freq}", f"--output_dir={output_dir}",
        f"--job_name={job_name}", "--seed=1000", "--wandb.enable=false",
    ]
    return [
        ACCELERATE, "launch", "--multi_gpu", "--num_processes=2",
        "--mixed_precision=no", "--main_process_port=29519", TRAIN_CLI, *train_args,
    ]

def run_and_log(command: list[str], log_path: Path) -> None:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    child_env = os.environ.copy()
    child_env["TQDM_DISABLE"] = "1"
    print("command:", subprocess.list2cmdline(command))
    with log_path.open("w", buffering=1) as log_file:
        process = subprocess.Popen(
            command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, env=child_env,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            log_file.write(line)
        return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f"Training exited with code {return_code}; inspect {log_path}")

In [ ]:
# Mandatory 20-update smoke from the retained 20k weights.
import re
import statistics

stamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
SMOKE_OUTPUT = WORK_ROOT / "outputs" / f"smoke_{stamp}"
SMOKE_LOG = WORK_ROOT / "logs" / f"smoke_{stamp}.log"
run_and_log(
    build_command(steps=20, output_dir=SMOKE_OUTPUT, job_name="smolvla_phase_c_extend_smoke", save_checkpoint=False, save_freq=20, log_freq=1),
    SMOKE_LOG,
)
smoke_text = SMOKE_LOG.read_text(errors="replace")
timings = [(float(update), float(data)) for update, data in re.findall(r"updt_s:([0-9.]+).*?data_s:([0-9.]+)", smoke_text)]
memory = [float(value) for value in re.findall(r"mem_gb:([0-9.]+)", smoke_text)]
losses = [float(value) for value in re.findall(r"loss:([0-9.]+)", smoke_text)]
assert timings and memory and losses, "Smoke completed without parseable finite metrics"
steady = timings[-min(5, len(timings)):]
seconds_per_step = statistics.mean(update + data for update, data in steady)
estimated_hours = seconds_per_step * EXTRA_STEPS / 3600
print(f"smoke passed: {seconds_per_step:.3f} s/step, peak logged memory {max(memory):.2f} GiB")
print(f"projected extra 60k update time: {estimated_hours:.2f} h")

In [ ]:
# Add 60k updates from the same retained 20k weights; smoke weights are not reused.
full_stamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
FULL_OUTPUT = WORK_ROOT / "outputs" / f"smolvla_phase_c_extend_20k_to_80k_{full_stamp}"
FULL_LOG = WORK_ROOT / "logs" / f"smolvla_phase_c_extend_20k_to_80k_{full_stamp}.log"
full_command = build_command(
    steps=EXTRA_STEPS, output_dir=FULL_OUTPUT, job_name="smolvla_phase_c_extend_20k_to_80k",
    save_checkpoint=True, save_freq=10_000, log_freq=50,
)
run_and_log(full_command, FULL_LOG)
manifest = {
    "completed_at": dt.datetime.now(dt.timezone.utc).isoformat(),
    "lerobot_commit": LEROBOT_COMMIT, "dataset_id": DATASET_ID,
    "dataset_revision": DATASET_REVISION, "source_model": str(MODEL_ROOT),
    "starting_total_steps": START_TOTAL_STEPS, "extra_steps": EXTRA_STEPS,
    "final_total_steps": START_TOTAL_STEPS + EXTRA_STEPS,
    "per_gpu_batch": PER_GPU_BATCH, "global_batch": PER_GPU_BATCH * 2,
    "optimizer_lr": 1e-5, "scheduler_decay_lr": 2.5e-6,
    "scheduler_warmup_steps": 500, "scheduler_decay_steps": EXTRA_STEPS,
    "gpus": gpu_names, "smoke_seconds_per_step": seconds_per_step,
    "smoke_projected_extra_60k_hours": estimated_hours,
    "output_dir": str(FULL_OUTPUT), "log_path": str(FULL_LOG), "command": full_command,
}
manifest_path = WORK_ROOT / "smolvla_phase_c_extend_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
print("manifest:", manifest_path)

In [ ]:
checkpoint_models = sorted(
    model_path for model_path in (FULL_OUTPUT / "checkpoints").glob("*/pretrained_model/model.safetensors")
    if model_path.parents[1].name.isdigit()
)
assert len(checkpoint_models) == 6, f"Expected six extension checkpoints, found {len(checkpoint_models)}"
for model_path in checkpoint_models:
    extension_step = int(model_path.parents[1].name)
    print("cumulative", START_TOTAL_STEPS + extension_step, "steps:", model_path.relative_to(WORK_ROOT), f"{model_path.stat().st_size / 2**20:.1f} MiB")
print("full log:", FULL_LOG)